### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 20
LEVEL = "high"
SEED = 42

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

print("-"*100); print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=20
✅ 시뮬레이션 초기화 완료: S=20, Randomness='high', Random Seed=42, M1=763.86, M2=2075.61
----------------------------------------------------------------------------------------------------
[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  9.35it/s]

Optimal solution found for target_i=0! Objective value: 242168.74115753826
Set parameter MIPGap to value 1e-07
Optimal solution found for target_i=1! Objective value: 357784.10700749425
Set parameter MIPGap to value 1e-07


...

Optimal solution found for target_i=2! Objective value: 409959.5100997424
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  8.77it/s]

Optimal solution found for target_i=3! Objective value: 463735.6473019376
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00,  9.02it/s]

Optimal solution found for target_i=4! Objective value: 180377.19278744285


### Linear Decision Rule (Holistic Optimization)

In [2]:
model = gp.Model("holistic_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 3)

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 3


In [3]:
x_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
ym_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") 
dm_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc")
zd_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
phi1_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")

model.update()

obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
model.setObjective(obj, GRB.MAXIMIZE)

# # Quadratic Regularization
# obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
#        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
# reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
# epsilon = 1e-7
# regularized_obj = obj - epsilon * reg
# model.setObjective(regularized_obj, GRB.MAXIMIZE)


In [4]:
for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    model.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
    model.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
    model.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
    model.addConstr(z_hol[i, t, s] <= K[i])
    model.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        
    model.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
    model.addConstr(dp_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
    model.addConstr(yp_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
    model.addConstr(dm_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
    
for i, s in product(range(I), range(S)): model.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = model.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

model.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  3
PoolSearchMode  2

Optimize a model with 43780 rows, 31420 columns and 110500 nonzeros
Model fingerprint: 0x2606bfd8
Variable types: 17020 continuous, 14400 integer (14400 binary)
Coefficient statistics:
  Matrix range     [9e-01, 8e+02]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 8e+02]
Presolve removed 11340 rows and 4410 columns
Presolve time: 0.08s
Presolved: 32440 rows, 27010 columns, 84056 nonzeros
Variable types: 12610 continuous, 14400 integer (14400 binary)
Found heuristic solution: objective 1722385.4750

Root relaxation: objective 1.760965e+06, 19393 iterations, 0.35 seconds (0.63 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth 

In [20]:
if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference from best = {diff:.8f}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    phi1_sol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_sol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi3_sol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi4_sol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi5_sol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi6_sol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])


--- Solution Pool Analysis ---
Found 3 solutions in the pool.
Best objective value: 1755976.09569415

Solution 0: Objective = 1755976.09569415,  Difference from best = 0.00000000
Solution 1: Objective = 1755976.09569415,  Difference from best = 0.00000000
Solution 2: Objective = 1755976.09569415,  Difference from best = 0.00000000


In [21]:
if model.status == GRB.OPTIMAL:        
    try:
        lambda_dual = {}
        for t, s in product(range(T), range(S)): 
            lambda_dual[t, s] = balance_constraints[t, s].Pi
        print("Direct dual extraction successful!")
        
    except AttributeError:
        print("\nDirect dual extraction failed. Using Model.fixed() method...")
        
        fixed_model = model.fixed()
        
        fixed_x_hol = {(i, t): fixed_model.getVarByName(f"x[{i},{t}]") for i, t in product(range(I), range(T))}
        fixed_yp_hol = {(i, t, s): fixed_model.getVarByName(f"yp[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}
        fixed_ym_hol = {(i, t, s): fixed_model.getVarByName(f"ym[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}

        linear_obj_for_fixed_model = (
            gp.quicksum(P_DA[t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T))) +
            gp.quicksum((1/S) * (P_RT[t, s] * fixed_yp_hol[i, t, s] - P_PN[t, s] * fixed_ym_hol[i, t, s]) 
                         for i, t, s in product(range(I), range(T), range(S)))
        )
        
        # reg_fixed = gp.quicksum(fixed_x_hol[i, t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T)))
        # regularized_obj_fixed = linear_obj_for_fixed_model - epsilon * reg_fixed
        regularized_obj_fixed = linear_obj_for_fixed_model
        
        fixed_model.setParam("OutputFlag", 0)
        fixed_model.setParam("MIPGap", 1e-5)
        fixed_model.setParam(GRB.Param.PoolSearchMode, 2)
        fixed_model.setParam(GRB.Param.PoolSolutions, 4)
        fixed_model.setObjective(regularized_obj_fixed, GRB.MAXIMIZE)
        fixed_model.optimize()
        
        if fixed_model.status == GRB.OPTIMAL:
            original_obj_val = (
                sum(P_DA[t] * x_sol[i, t] for i, t in product(range(I), range(T))) +
                sum((1/S) * (P_RT[t, s] * yp_sol[i, t, s] - P_PN[t, s] * ym_sol[i, t, s]) 
                    for i, t, s in product(range(I), range(T), range(S)))
                # - epsilon * sum(x_sol[i, t] * x_sol[i, t] for i, t in product(range(I), range(T)))
            )
            
            fixed_obj = fixed_model.objVal
            obj_diff = abs(original_obj_val - fixed_obj)

            print(f"Original Objective: {original_obj_val:.6f}")
            print(f"Fixed Model Objective: {fixed_obj:.6f}")
            print(f"Difference: {obj_diff:.10f}")
            
            if obj_diff < 1e-6:
                print("✅ Objective values match! Fixed model is consistent.")
            else:
                print("⚠️ Warning: Objective values don't match.")

            num_solutions = fixed_model.SolCount
            print(f"\n--- Fixed Model Solution Pool Analysis ---")
            print(f"Found {num_solutions} solutions in the pool.")

            if num_solutions > 1:
                best_obj = fixed_model.objVal
                print(f"Best objective value: {best_obj:.8f}\n")
                for i in range(num_solutions):
                    fixed_model.setParam(GRB.Param.SolutionNumber, i)
                    pool_obj = fixed_model.PoolObjVal
                    diff = abs(best_obj - pool_obj)
                    print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference = {diff:.8f}")
                fixed_model.setParam(GRB.Param.SolutionNumber, 0)

            print("\n=== Solution Comparison ===")
            fixed_vars = {var.VarName: var.X for var in fixed_model.getVars()}
            
            print("=== x values by individual ===")
            print("Individual | Time | Original x | Fixed x | Difference")
            print("-" * 55)
            
            max_x_diff = 0
            for i in range(I):
                for t in range(T):
                    original_x = x_sol[i, t]
                    fixed_x = fixed_vars.get(f"x[{i},{t}]", 0)
                    diff = abs(original_x - fixed_x)
                    max_x_diff = max(max_x_diff, diff)
                    if original_x != 0:
                        print(f"{i:10d} | {t:4d} | {original_x:10.6f} | {fixed_x:7.6f} | {diff:10.8f}")
            
            print("\n=== yp values sum over i (scenario average) ===")
            print("Time | Original yp_sum | Fixed yp_sum | Difference")
            print("-" * 52)
            max_yp_diff = 0
            for t in range(14,16):
                original_yp_sum = sum(sum(yp_sol[i, t, s] for i in range(I)) for s in range(S)) / S
                fixed_yp_sum = sum(sum(fixed_vars.get(f"yp[{i},{t},{s}]", 0) for i in range(I)) for s in range(S)) / S
                diff = abs(original_yp_sum - fixed_yp_sum)
                max_yp_diff = max(max_yp_diff, diff)
                print(f"{t:4d} | {original_yp_sum:14.6f} | {fixed_yp_sum:12.6f} | {diff:10.8f}")

        lambda_dual = np.zeros((T, S))
        if fixed_model.status == GRB.OPTIMAL:
            for t, s in product(range(T), range(S)):
                constr_name = f"balance_{t}_{s}"
                try:
                    constr = fixed_model.getConstrByName(constr_name)
                    if constr is not None:
                        lambda_dual[t, s] = constr.Pi
                    else:
                        lambda_dual[t, s] = np.nan
                except:
                    lambda_dual[t, s] = np.nan 
                    
            print("\nModel.fixed() dual extraction successful!")
        else:
            print("Fixed model optimization failed. Setting dual to zeros.")
            lambda_dual = np.zeros((T, S))

        print("\nInternal Settlement Prices (Dual Variables):")
        for t, s in product(range(T), range(min(1, S))):
            print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
            
else:
    print("No optimal solution found.")
    lambda_dual = {(t, s): np.nan for t in range(T) for s in range(S)}
    x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
    original_objval = None


Direct dual extraction failed. Using Model.fixed() method...
Original Objective: 1755976.095694
Fixed Model Objective: 1755976.095694
Difference: 0.0000000002
✅ Objective values match! Fixed model is consistent.

--- Fixed Model Solution Pool Analysis ---
Found 1 solutions in the pool.

=== Solution Comparison ===
=== x values by individual ===
Individual | Time | Original x | Fixed x | Difference
-------------------------------------------------------
         0 |    9 |  42.890655 | 42.890655 | 0.00000000
         0 |   16 | 292.367242 | 270.797138 | 21.57010443
         0 |   20 |  14.326179 | 17.670173 | 3.34399405
         1 |   10 |  23.478961 | 23.478961 | 0.00000000
         1 |   11 |  40.413464 | 40.413464 | 0.00000000
         1 |   14 |   0.165930 | 0.165930 | 0.00000000
         1 |   18 |  40.336818 | 40.336818 | 0.00000000
         1 |   20 |   1.058398 | 2.649637 | 1.59123898
         2 |   10 | 176.182386 | 176.182386 | 0.00000000
         2 |   11 | 227.737144 | 227.

In [26]:
data = []
for t, s in product(range(T), range(0,1)):
    data.append({
        'Time': t,
        'Scenario': s,
        'P_DA': round(P_DA[t], 2),
        'P_RT': round(P_RT[t, s], 2),
        'Lambda': round(-lambda_dual[t, s] * S, 2),
        'P_PN': round(P_PN[t, s], 2)
    })
pd.DataFrame(data)

,Time,Scenario,P_DA,P_RT,Lambda,P_PN
0,0,0,106.720,58.820,-0.000,213.440
1,1,0,93.130,97.470,-0.000,194.930
2,2,0,86.030,55.420,-0.000,172.060
3,3,0,82.960,63.350,-0.000,165.930
4,4,0,82.730,91.930,118.270,183.860
5,5,0,85.950,57.880,135.320,171.890
6,6,0,93.550,86.780,-0.000,187.100
7,7,0,102.330,118.270,129.160,236.540
8,8,0,122.120,95.570,-0.000,244.240
9,9,0,132.150,88.630,117.340,264.290


In [36]:
lambda_rep = np.zeros((T, S))

data = []
for t in range(T):
    for s in range(S):
        lambda_rep[t, s] = lambda_dual[t, s]
    p_rt_avg = np.mean(P_RT[t, :]) ; p_pn_avg = np.mean(P_PN[t, :])
    
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :])
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :][lambda_rep[t, :] < 0]) if np.any(lambda_rep[t, :] < 0) else 0
    lambda_rep[t, :] = np.mean(lambda_rep[t, :][np.abs(lambda_rep[t, :]) > 0]) if np.any(np.abs(lambda_rep[t, :]) > 0) else 0
    
    data.append({
        'Time': t, 'P_RT_avg': round(p_rt_avg, 2), 'Lambda': round(-lambda_rep[t, 0] * S, 2), 'P_PN_avg': round(p_pn_avg, 2)
    })
pd.DataFrame(data)

,Time,P_RT_avg,Lambda,P_PN_avg
0,0,62.570,-0.000,213.440
1,1,87.530,-0.000,196.750
2,2,74.840,-0.000,178.380
3,3,66.830,-0.000,167.120
4,4,72.650,133.210,168.650
5,5,86.840,148.090,188.890
6,6,71.170,142.360,187.240
7,7,86.800,32.020,209.710
8,8,100.300,-0.000,246.570
9,9,106.080,132.150,264.290


### Individual Replay

In [54]:
def individual_replay(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, lambda_dual, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol):
    
    model = gp.Model("DER_Individual_Replay")
    model.setParam("MIPGap", 1e-4)
    model.setParam(GRB.Param.PoolSearchMode, 2)
    model.setParam(GRB.Param.PoolSolutions, 4)
    
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    # phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
    # phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    # phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    
    model.update()

    obj = (
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S)) +
        gp.quicksum(lambda_dual[t, s] * (
            gp.quicksum(dm[i, t, s] for i in range(I)) - gp.quicksum(dp[i, t, s] for i in range(I))
        ) for t in range(T) for s in range(S))
    )
    epsilon = 1e-7
    reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
    
    regularized_obj = obj - epsilon * reg
    
    model.setObjective(regularized_obj, GRB.MAXIMIZE)
    # model.setObjective(obj, GRB.MAXIMIZE)
    
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s]) ; model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s]) ; model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.92 * zc[i, t, s] - zd[i, t, s] / 0.95)
        
        model.addConstr(yp[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        model.addConstr(dm[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
        
        # model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
        # model.addConstr(dp[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2[i, t, s]))
        # model.addConstr(yp[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3[i, t, s]))
        # model.addConstr(ym[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4[i, t, s]))
        # model.addConstr(ym[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5[i, t, s]))
        # model.addConstr(dm[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6[i, t, s]))
    

    for i, s in product(range(I), range(S)): model.addConstr(z[i, 0, s] == K0[i])

    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        num_solutions = model.SolCount
        print(f"\n--- Solution Pool Analysis ---")
        print(f"Found {num_solutions} solutions in the pool.")

        if num_solutions > 1:
            best_obj = model.objVal
            print(f"Best objective value: {best_obj:.8f}\n")

            for i in range(num_solutions):
                model.setParam(GRB.Param.SolutionNumber, i)
                pool_obj = model.PoolObjVal
                diff = best_obj - pool_obj
                
                print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference from best = {diff:.8f}")

        model.setParam(GRB.Param.SolutionNumber, 0)
        
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    return (x_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, dp_sol, dm_sol, model.objVal)


In [55]:
x_re, yp_re, ym_re, z_re, zc_re, zd_re, dp_re, dm_re, OBJ_RE = individual_replay(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, lambda_rep, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol)

Set parameter MIPGap to value 0.0001
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 4
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G84)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
PoolSolutions  4
PoolSearchMode  2

Optimize a model with 43300 rows, 17020 columns and 76900 nonzeros
Model fingerprint: 0x77666018
Model has 120 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [2e+00, 2e+02]
  QObjective range [2e-07, 2e-07]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e-02, 8e+02]
Presolve removed 37217 rows and 12192 columns
Presolve time: 0.01s
Presolved: 6083 rows, 4828 columns, 24938 nonzeros
Presolved model has 88 quadratic objective terms
Ordering time: 0.01s

Barrier statistics:
 Dense cols : 16
 AA' NZ     : 4.476e+04
 Factor NZ  : 1.673e+05 (roughly 6 MB of memory)
 Factor Ops 

In [56]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[REPLAY]") ; print(header)
for t in range(9, 20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

print("\n[HOLISTIC]") ; print(header)
for t in range(9, 20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 9 |   185.31    24.47    26.38     0.00     0.00     0.00   135.01     0.55    47.39
10 |   445.25   213.52    78.16     0.00    38.98    31.55   156.80    10.66   171.02
11 |   652.06   308.16   185.49     0.00    65.18    55.30   149.19     0.66   304.05
12 |   860.69     0.00   844.92     0.00     0.00     0.00    49.84    34.07   440.61
13 |  1397.66     0.00  1756.93     0.00     0.00     0.00     2.23   361.50   450.60
14 |  1338.06   831.78   242.04     0.00   585.08   649.54   328.70     0.00    72.12
15 |   804.05     0.00  1095.14     0.00     0.00     0.00     8.14   299.23   374.52
16 |   756.56   299.69    78.24     0.00   290.12    29.20   158.43    40.72    67.03
17 |   738.45     0.00   825.81     0.00     0.00     0.00    40.00   127.36   169.93
18 |   493.98   302.06   153.75     0.0

In [58]:
import numpy as np

# Aggregator 손실 계산
print("="*50)
print("AGGREGATOR LOSS ANALYSIS")
print("="*50)

total_losses = []

for t in range(T):
    scenario_losses = []
    
    for s in range(S):
        # 각 시나리오별 total supply/demand
        total_supply = np.sum(dp_re[:, t, s])  # sum d+
        total_demand = np.sum(dm_re[:, t, s])  # sum d-
        
        if total_supply > total_demand:
            # Excess supply: (sum d+ - sum d-) * (lambda - P_RT) 손실
            excess = total_supply - total_demand
            # loss = excess * (-lambda_rep[t, s]*S - P_RT[t, s])
            loss = excess * (P_RT[t, s])
        else:
            # Excess demand: (sum d- - sum d+) * (P_PN - lambda) 손실
            excess = total_demand - total_supply
            # loss = excess * (P_PN[t, s] - (-lambda_rep[t,s]*S))
            loss = excess * (P_PN[t, s])
        
        scenario_losses.append(loss)
        
        # # 각 시나리오 출력
        # print(f"t={t}, s={s}: supply={total_supply:.2f}, demand={total_demand:.2f}, "
        #       f"excess={'supply' if total_supply > total_demand else 'demand'}={abs(total_supply-total_demand):.2f}, "
        #       f"loss={loss:.2f}")
    
    # s에 대한 평균
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)
    
    # print(f"t={t} Average Loss: {avg_loss:.2f}")
    # print()

# 전체 평균 손실
overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

print("[SUMMARY]")
print("Individual Participation Profit", sum(OBJ_IND[i] for i in range(I)))
print("Expected Replay Profit", OBJ_RE)
print(f"Total loss across all time periods: {total_loss:.2f}")
print("Realized Profit", OBJ_RE - total_loss)
print("Holistic Profit", OBJ_HOL)

AGGREGATOR LOSS ANALYSIS
[SUMMARY]
Individual Participation Profit 1654025.1983541553
Expected Replay Profit 1755261.002062462
Total loss across all time periods: 124522.70
Realized Profit 1630738.3037407123
Holistic Profit 1755261.090266614
